![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 03: Big Data)**

**Session 3D: Data Acquisition I**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 03. The full module is normally completed across two two-hour practical sessions, together with the other notebooks listed in the SIT742 repository.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with Python, NumPy, and public SIT742 data files</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>Portable NumPy and file I/O workflow for TXT, CSV, and JSON data</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development; not directly assessed</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Data Files](#2-setup-and-data-files)
- [3. Background Concepts](#3-background-concepts)
- [4. Guided Examples](#4-guided-examples)
- [5. Practical Exercises](#5-practical-exercises)
- [6. Student Tasks](#6-student-tasks)
- [7. Reflection and References](#7-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>
### 1. Overview and Learning Goals

Data acquisition starts with two practical questions: how should data be represented in memory, and how should files be loaded or written in a repeatable way? This session introduces NumPy arrays for efficient numerical work and then practises loading and saving TXT, CSV, and JSON files from public SIT742 data sources.

By the end of this lab, students should be able to:

1. create, inspect, index, slice, reshape, and combine NumPy arrays;
2. explain why vectorised array operations are usually preferred over manual loops;
3. load public TXT, CSV, and JSON files in both online and local notebook environments;
4. write generated files into a controlled runtime output folder; and
5. use lightweight checks to verify data shapes, types, and file paths.


<a id="2-setup-and-data-files"></a>
### 2. Setup and Data Files

This notebook uses three small public SIT742 files: `txt_data1.txt`, `csv_data1.csv`, and `json_data1.json`.

#### Option A: Google Colab / online execution

Keep `EXECUTION_MODE = "online"`. The setup cell downloads public data files from GitHub into this notebook runtime.

#### Option B: Local repository execution

Use `EXECUTION_MODE = "local"` only when you have cloned the public SIT742 repository and are running the notebook from a location where the `Jupyter/data/` folder can be found.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import json
import sys
import tempfile

import numpy as np

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.

required_files = ["txt_data1.txt", "csv_data1.csv", "json_data1.json"]
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sit742_m03d_output_"))


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m03d_data_"))
    downloaded_paths = {}
    for filename in required_files:
        url = f"{PUBLIC_DATA_BASE_URL}/{filename}"
        local_file = data_dir / filename
        urlretrieve(url, local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    candidates = [
        Path.cwd() / "Jupyter" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Jupyter" / "data",
    ]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the SIT742 public data folder. Searched:\n" + searched
    )


if EXECUTION_MODE == "online":
    DATA_DIR, data_paths_by_file = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR = find_local_data_dir(required_files)
    data_paths_by_file = {filename: DATA_DIR / filename for filename in required_files}
else:
    raise ValueError('EXECUTION_MODE must be "online" or "local"')

data_paths = {
    "txt": data_paths_by_file["txt_data1.txt"],
    "csv": data_paths_by_file["csv_data1.csv"],
    "json": data_paths_by_file["json_data1.json"],
}

# Backward-compatible file-path variables for students comparing with older examples.
DataSetTXT = data_paths["txt"]
DataSetCSV = data_paths["csv"]
DataSetJSON = data_paths["json"]
DataSet = DataSetCSV

print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("Execution mode:", EXECUTION_MODE)
print("Data folder:", DATA_DIR)
print("Temporary output folder:", OUTPUT_DIR)


In [ ]:
for label, path in data_paths.items():
    print(f"{label}: {path.name} ({path.stat().st_size} bytes)")


<a id="3-background-concepts"></a>
### 3. Background Concepts

NumPy adds efficient array objects and vectorised operations to Python. Arrays are less flexible than ordinary Python lists, but they are a better fit for many numerical tasks because operations can be applied to whole arrays at once.

This notebook also uses three common acquisition formats:

- **TXT**: simple text data, useful for small numeric examples and line-based text;
- **CSV**: comma-separated values, widely used for tabular data;
- **JSON**: nested key-value data, common in web services and APIs.

The goal is not only to load these files once, but to make the loading path portable across Colab and local repository execution.


<a id="4-guided-examples"></a>
### 4. Guided Examples

#### 4.1 Creating NumPy arrays

Start with a Python list, convert it to a NumPy array, and inspect the array type.


In [ ]:
x = [1, 7, 3, 4, 0, -5]
y = np.array(x)

print("Python list:", x)
print("NumPy array:", y)
print("Array type:", type(y))


NumPy has several constructors for creating numeric sequences. Compare `arange`, `linspace`, and `logspace` carefully: they answer different spacing questions.


In [ ]:
print("range as array:", np.array(range(5)))
print("arange:", np.arange(2, 3, 0.2))
print("linspace:", np.linspace(2, 3, 5))
print("logspace:", np.logspace(2, 3, 5))

# In an interactive notebook, run help(np.logspace) or np.logspace? if you want
# to inspect the full documentation.


#### 4.2 Prefilled arrays, grids, and attributes


In [ ]:
print("zeros:", np.zeros(5))
print("ones vector:", np.ones(5, dtype=int))
print("ones matrix:\n", np.ones((3, 4), dtype=int))

x_grid, y_grid = np.mgrid[0:5, 0:3]
print("x grid:\n", x_grid)
print("y grid:\n", y_grid)

sample = np.array([3, 0, -4, 6, 12, 2])
print("number of dimensions:", sample.ndim)
print("shape:", sample.shape)
print("data type:", sample.dtype)
print("maximum:", sample.max())
print("index of maximum:", sample.argmax())
print("mean:", sample.mean())


#### 4.3 Multi-dimensional arrays, indexing, and masks


In [ ]:
matrix = np.array([[7, 6, 8, 6, 4], [4, 7, -2, 0, 9]])

print("matrix:\n", matrix)
print("row 1, column 3:", matrix[1, 3])
print("second row:", matrix[1, :])
print("fourth column:", matrix[:, 3])

values = np.array([2, 8, -2, 4, 3, 9, 0])
mask = (values >= 2) & (values < 9)
print("mask:", mask)
print("selected values:", values[mask])


#### 4.4 Slicing, iteration, and copying


In [ ]:
values = np.array([2, 8, -2, 4, 3, 9, 0])
print("slice from index 3:", values[3:])
print("slice with step:", values[3:7:2])

grid = np.array([[7, 6, 8, 6, 4, 3], [4, 7, 0, 5, 9, 5], [7, 3, 6, 3, 5, 1]])
print("row slice:", grid[1, 1:4])
print("2D slice:\n", grid[:2, 1::2])

original = np.array([1, 2, 3])
alias = original
alias[0] = 0
print("After alias edit:", original, alias)

original = np.array([1, 2, 3])
copy_of_original = original.copy()
copy_of_original[0] = 0
print("After copy edit:", original, copy_of_original)


#### 4.5 Reshaping, type casting, and transposing


In [ ]:
one_dimensional = np.arange(6)
two_dimensional = one_dimensional.reshape((2, 3))
print("1D:", one_dimensional)
print("2D:\n", two_dimensional)

integers = np.arange(5)
floats = integers.astype(float)
print("integer dtype:", integers.dtype)
print("float dtype:", floats.dtype)

rng = np.random.default_rng(742)
random_matrix = rng.integers(0, 5, size=(2, 4))
print("matrix:\n", random_matrix)
print("transpose:\n", random_matrix.T)


#### 4.6 Array operations and random numbers


In [ ]:
x1 = np.array([[2, 3, 5, 7], [2, 4, 6, 8]], dtype=float)
x2 = np.array([[6, 5, 4, 3], [9, 7, 5, 3]], dtype=float)

print("x1 + x2:\n", x1 + x2)
print("x1 - x2:\n", x1 - x2)
print("x1 * x2:\n", x1 * x2)
print("x1 / x2:\n", x1 / x2)
print("3 / x1:\n", 3 / x1)

comparison = np.array([2, 3, 5, 7]) < np.array([2, 4, 6, 7])
print("comparison:", comparison)
print("all true?", comparison.all())
print("any true?", comparison.any())

rng = np.random.default_rng(100)
print("reproducible random values:", rng.random(5))


#### 4.7 Vectorising scalar logic


In [ ]:
def step_func(value):
    if value >= 0:
        return 1
    return 0


try:
    step_func(np.array([2, 7, -4, -9, 0, 4]))
except ValueError as exc:
    print("The scalar version does not work on an array:")
    print(exc)

step_func_vectorised = np.vectorize(step_func)
print("vectorised:", step_func_vectorised(np.array([2, 7, -4, -9, 0, 4])))


def step_func_array(values):
    return 1 * (values >= 0)


print("array-aware:", step_func_array(np.array([2, 7, -4, -9, 0, 4])))


#### 4.8 Loading and saving TXT files

`np.loadtxt()` is useful for simple numeric text files. Write generated files to `OUTPUT_DIR` so rerunning the notebook does not clutter the repository folder.


In [ ]:
txt_values = np.loadtxt(data_paths["txt"])
print("TXT values:", txt_values)

generated_txt = OUTPUT_DIR / "txt_data2.txt"
new_values = np.random.default_rng(742).integers(0, 10, size=5)
np.savetxt(generated_txt, new_values, fmt="%d")

print("Saved TXT file:", generated_txt)
print("Reloaded values:", np.loadtxt(generated_txt))


#### 4.9 Loading and saving CSV files


In [ ]:
csv_array = np.genfromtxt(data_paths["csv"], delimiter=",")
print("CSV array shape:", csv_array.shape)
print(csv_array)

generated_csv = OUTPUT_DIR / "csv_data2.csv"
new_matrix = np.random.default_rng(742).integers(0, 10, size=(6, 4))
np.savetxt(generated_csv, new_matrix, delimiter=",", fmt="%d")

print("Saved CSV file:", generated_csv)
print(np.genfromtxt(generated_csv, delimiter=","))


#### 4.10 Loading and saving JSON files


In [ ]:
with data_paths["json"].open("r", encoding="utf-8") as fp:
    json_data = json.load(fp)

print("JSON keys:", sorted(json_data.keys()))
print("First phone entry:", json_data["phoneNumbers"][0])

generated_json = OUTPUT_DIR / "json_data_now.json"
records = [
    {"Name": "Zara", "Age": 7, "Class": "First"},
    {"Name": "Lily", "Age": 9, "Class": "Third"},
]
with generated_json.open("w", encoding="utf-8") as fp:
    json.dump(records, fp, indent=2)

print("Saved JSON file:", generated_json)
print("Reloaded record count:", len(json.loads(generated_json.read_text())))


<a id="5-practical-exercises"></a>
### 5. Practical Exercises

Use the checks below to verify that the core workflow is behaving as expected. These checks focus on file availability, array shape, and generated output paths.


In [ ]:
assert txt_values.ndim == 1
assert csv_array.shape == (6, 4)
assert "phoneNumbers" in json_data
assert generated_txt.exists()
assert generated_csv.exists()
assert generated_json.exists()

print("All checks passed.")


Try modifying the array and file examples:

1. Change the `linspace` call to create 9 values instead of 5.
2. Create a mask that selects only negative values from `values`.
3. Add a third record to the JSON `records` list and write it to a new output file.


In [ ]:
# Student workspace
# 1. Create a new linspace array.
# 2. Build a boolean mask for negative values.
# 3. Add a new JSON record and save it in OUTPUT_DIR.


<a id="6-student-tasks"></a>
### 6. Student Tasks

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td>Create a 3 by 4 NumPy array and report its shape, data type, minimum, maximum, and mean.</td>
<td>Array inspection is the first check before numeric analysis.</td>
<td>A code cell with printed summary values.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td>Use a boolean mask to select values from an array according to a rule you define.</td>
<td>Masking is a common pattern for filtering data.</td>
<td>The rule, mask, and selected values.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td>Load one public data file and write one generated output file into `OUTPUT_DIR`.</td>
<td>Portable data loading and controlled outputs make notebooks easier to rerun.</td>
<td>The input path, output path, and a short shape or record-count check.</td>
</tr>
</tbody>
</table>

</div>


<a id="7-reflection-and-references"></a>
### 7. Reflection and References

Reflection prompts:

1. When would a NumPy array be a better choice than a Python list?
2. What can go wrong when a notebook writes files into the current working directory?
3. How does the online/local setup pattern help students who have not cloned the repository?

Further readings:

- NumPy user guide: <https://numpy.org/doc/stable/user/>
- NumPy file I/O: <https://numpy.org/doc/stable/reference/routines.io.html>
- Python `json` documentation: <https://docs.python.org/3/library/json.html>
- Python `urllib.request` documentation: <https://docs.python.org/3/library/urllib.request.html>
- Public SIT742 repository: <https://github.com/tulip-lab/sit742>
